<a href="https://colab.research.google.com/github/nkefeyan-22-26/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Lab12/Lab%2012%3A%20Architecting%20the%20Prediction%20Engine_OLS%2C%20Hedonic%20Pricing%2C%20and%20RMSE%20Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
url = "https://raw.githubusercontent.com/nkefeyan-22-26/ECON5200-Applied-Data-Analytics-in-Economics/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv"

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
df = pd.read_csv(url)

In [6]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [7]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula,data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Tue, 10 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        18:53:55   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [8]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [9]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [10]:
# =============================================================================
# INTERACTIVE RESIDUAL FORENSICS DASHBOARD
# Hedonic Pricing OLS Model — Statsmodels + Plotly Express
# =============================================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go

# -----------------------------------------------------------------------------
# STEP 1: BUILD YOUR OLS MODEL (swap in your own DataFrame / variables)
# -----------------------------------------------------------------------------

# --- Synthetic hedonic dataset (replace with your actual data) ---
np.random.seed(42)
n = 300
df = pd.DataFrame({
    "sqft":      np.random.randint(500, 4000, n),
    "bedrooms":  np.random.randint(1, 6, n),
    "age":       np.random.randint(1, 60, n),
    "distance":  np.random.uniform(0.5, 30, n),   # miles from CBD
})

# Inject deliberate heteroscedasticity: variance grows with sqft
noise = np.random.normal(0, df["sqft"] * 0.05, n)
df["price"] = (
    150_000
    + 120   * df["sqft"]
    + 8_000 * df["bedrooms"]
    - 500   * df["age"]
    - 2_000 * df["distance"]
    + noise
)

# --- Design matrix with constant for OLS intercept ---
X = sm.add_constant(df[["sqft", "bedrooms", "age", "distance"]])
y = df["price"]

# --- Fit OLS ---
model   = sm.OLS(y, X)
results = model.fit()

print(results.summary())

# -----------------------------------------------------------------------------
# STEP 2: EXTRACT RESIDUALS & FITTED VALUES FROM STATSMODELS RESULTS OBJECT
# -----------------------------------------------------------------------------

# `results.fittedvalues` → pandas Series of ŷ (predicted values on training data).
# Statsmodels computes these internally as X @ β̂.
fitted    = results.fittedvalues          # ŷ — x-axis of our plot

# `results.resid` → pandas Series of raw residuals: e = y − ŷ.
# These are NOT standardised; they live in the original dollar scale.
residuals = results.resid                 # e — y-axis of our plot

# -----------------------------------------------------------------------------
# STEP 3: CLASSIFY OUTLIERS (|z-score| > 2 SD)
# -----------------------------------------------------------------------------

# Compute the standard deviation of the residual distribution.
resid_std  = residuals.std()              # σ̂ of residuals

# Compute a z-score for every residual to measure how many SDs it sits from 0.
# We use residuals (not standardised residuals from results.get_influence()),
# keeping this self-contained and transparent.
resid_zscore = (residuals - residuals.mean()) / resid_std

# Boolean mask: True wherever the absolute z-score exceeds 2.
is_outlier = resid_zscore.abs() > 2      # True  → outlier, False → normal

# Map the boolean mask to human-readable category labels.
# Plotly Express will use these labels for colour encoding.
outlier_label = is_outlier.map({True: "Outlier (|z| > 2)", False: "Normal"})

# -----------------------------------------------------------------------------
# STEP 4: ASSEMBLE PLOTTING DATAFRAME
# -----------------------------------------------------------------------------

plot_df = pd.DataFrame({
    "Fitted Values"  : fitted,
    "Residuals"      : residuals,
    "Z-Score"        : resid_zscore.round(3),
    "Observation"    : df.index,
    "Status"         : outlier_label,
})

# -----------------------------------------------------------------------------
# STEP 5: PLOTLY EXPRESS SCATTER — FITTED vs RESIDUALS
# -----------------------------------------------------------------------------

# Discrete colour map: normal points in steel blue, outliers in crimson.
color_map = {
    "Normal"             : "#4A90D9",   # steel blue
    "Outlier (|z| > 2)"  : "#DC143C",   # stark crimson
}

fig = px.scatter(
    plot_df,
    x            = "Fitted Values",
    y            = "Residuals",
    color        = "Status",             # colour driven by our outlier labels
    color_discrete_map = color_map,
    hover_data   = {"Z-Score": True, "Observation": True},
    opacity      = 0.75,
    title        = "Residual Forensics Dashboard — Hedonic Pricing OLS",
    labels       = {
        "Fitted Values" : "Fitted Values  ŷ  ($)",
        "Residuals"     : "Residuals  e = y − ŷ  ($)",
    },
)

# -----------------------------------------------------------------------------
# STEP 6: ADD REFERENCE LINES & BAND ANNOTATIONS
# -----------------------------------------------------------------------------

# --- Horizontal zero-line: residuals should be randomly scattered around 0 ---
fig.add_hline(
    y           = 0,
    line_width  = 1.8,
    line_dash   = "solid",
    line_color  = "black",
    annotation_text     = "Zero baseline",
    annotation_position = "bottom right",
)

# --- ±1 SD band: dashed, subtle grey ---
for sign, label in [(1, "+1 SD"), (-1, "−1 SD")]:
    fig.add_hline(
        y          = sign * resid_std,
        line_width = 1,
        line_dash  = "dot",
        line_color = "grey",
        annotation_text     = label,
        annotation_position = "bottom right" if sign == -1 else "top right",
    )

# --- ±2 SD band: dashed, crimson to match outlier colour ---
for sign, label in [(2, "+2 SD  (outlier threshold)"), (-2, "−2 SD  (outlier threshold)")]:
    fig.add_hline(
        y          = sign * resid_std,
        line_width = 1.2,
        line_dash  = "dash",
        line_color = "#DC143C",
        annotation_text     = label,
        annotation_position = "bottom right" if sign < 0 else "top right",
    )

# -----------------------------------------------------------------------------
# STEP 7: LAYOUT POLISH
# -----------------------------------------------------------------------------

fig.update_layout(
    template        = "plotly_white",
    font            = dict(family="Arial", size=13),
    title_font_size = 18,
    legend_title    = "Observation Status",
    legend          = dict(
        orientation = "h",
        yanchor     = "bottom",
        y           = 1.02,
        xanchor     = "right",
        x           = 1,
    ),
    xaxis = dict(showgrid=True, gridcolor="#E8E8E8"),
    yaxis = dict(showgrid=True, gridcolor="#E8E8E8", zeroline=False),
    hoverlabel = dict(bgcolor="white", font_size=12),
    margin = dict(l=60, r=60, t=80, b=60),
)

fig.update_traces(marker=dict(size=7, line=dict(width=0.5, color="white")))

fig.show()

# -----------------------------------------------------------------------------
# STEP 8: PRINT OUTLIER SUMMARY
# -----------------------------------------------------------------------------

n_outliers = is_outlier.sum()
pct        = n_outliers / n * 100

print(f"\n{'='*55}")
print(f"  Residual Forensics Summary")
print(f"{'='*55}")
print(f"  Total observations     : {n}")
print(f"  Residual std dev (σ̂)  : ${resid_std:,.2f}")
print(f"  Outliers (|z| > 2 SD) : {n_outliers}  ({pct:.1f}%)")
print(f"  Expected under normality: ~{n * 0.0456:.0f}  (4.56%)")
print(f"{'='*55}")

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 7.133e+07
Date:                Tue, 10 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:00:32   Log-Likelihood:                -1876.2
No. Observations:                 300   AIC:                             3762.
Df Residuals:                     295   BIC:                             3781.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.501e+05     30.291   4954.043      0.0


  Residual Forensics Summary
  Total observations     : 300
  Residual std dev (σ̂)  : $126.05
  Outliers (|z| > 2 SD) : 20  (6.7%)
  Expected under normality: ~14  (4.56%)
